# Notebook 47 — Cross-Regime Coefficient Transport and Universality

This notebook tests whether the sparse governing law extracted in Notebook 46 is **universal in structure**
and how its coefficients **transport across regimes**.

We lock the template family and fit comparable coefficient vectors across:

- `k`
- forcing mode
- task
- system

## Governing template family

We use the persistent term family suggested by Notebook 46:

\[
g(r,c) \approx b_0 + b_1 c + b_2 r + b_3 c^3 + b_4 r^2 + b_5 r c^2
\]

## Main questions

1. Does this term family remain adequate across regimes?
2. How do coefficients move across `k`, forcing mode, task, and system?
3. Can one regime's coefficients transfer to another regime?
4. Is there a low-dimensional manifold of coefficient transport?
5. How do coefficients relate to dynamical diagnostics such as asymmetry and additive fraction?

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.04, 5: 0.02, 7: 0.05}[k]
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                        "jacobian_norm": abs(-0.78 + 0.40 * r + nl_gain * 0.10 * c - 0.075 * r**2),
                                        "integration_error": abs(noise),
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")

    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

## Locked template family

We fit the same explicit template in every regime:

- intercept
- `c`
- `r`
- `c^3`
- `r^2`
- `r c^2`

This makes coefficient vectors directly comparable.

In [ ]:
# Fixed template design

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    X = np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])
    return X

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta

    out = {
        "n": len(sub),
        "r2": r2_score(y, pred),
        "rmse": math.sqrt(mean_squared_error(y, pred)),
    }
    for name, coef in zip(TERM_NAMES, beta):
        out[f"coef_{name}"] = float(coef)
    return beta, pred, out

def predict_with_beta(sub, beta):
    X = design_template(sub)
    return X @ beta

def additive_fraction(sub):
    g = sub[["condition_coord", "residual", "predicted_flow"]].dropna().copy()
    g["c_bin"] = pd.cut(g["condition_coord"], bins=20, labels=False, include_lowest=True)
    g["r_bin"] = pd.cut(g["residual"], bins=20, labels=False, include_lowest=True)
    grouped = g.groupby(["c_bin", "r_bin"]).agg(mean_flow=("predicted_flow", "mean")).reset_index()
    grid = grouped.pivot(index="r_bin", columns="c_bin", values="mean_flow").sort_index()
    G = grid.values
    overall = np.nanmean(G)
    col_eff = np.nanmean(G - overall, axis=0)
    row_eff = np.nanmean(G - overall, axis=1)
    G_add = overall + row_eff[:, None] + col_eff[None, :]
    denom = np.nansum((G - np.nanmean(G))**2)
    if denom <= 0:
        return np.nan
    return 1 - np.nansum((G - G_add)**2) / denom

def fb_gap_metric(sub, beta, n_r0=20):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 50)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)

    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def g_of(r, c):
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        val = float(x @ beta)
        return float(np.clip(val, -flow_cap, flow_cap))

    def integrate(r0, direction="forward"):
        use_c = cgrid if direction == "forward" else cgrid[::-1]
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(use_c) - 1):
            c = use_c[i]
            dc = float(use_c[i+1] - use_c[i])
            r = float(r + g_of(r, c) * dc)
            r = float(np.clip(r, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        traj = np.array(traj)
        if direction == "backward":
            traj = traj[::-1]
        return traj

    f_terms, b_terms = [], []
    for r0 in r0s:
        tf = integrate(r0, "forward")
        tb = integrate(r0, "backward")
        f_terms.append(tf[-1])
        b_terms.append(tb[0])
    return float(np.mean(np.abs(np.array(f_terms) - np.array(b_terms))))

## Focus slice and coefficient transport across `k`

In [ ]:
# Focus slice like Notebook 46

focus_mask = (
    (df["system"].astype(str) == "entropy")
    & (df["task"].astype(str) == "zeta_vs_gue")
    & (df["forcing_mode"].astype(str) == "condition_gap")
)

focus_candidates = df[focus_mask].copy()
if focus_candidates.empty:
    group_cols = ["system", "task", "forcing_mode", "flow_mode"]
    top_group = (
        df.groupby(group_cols)
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
        .iloc[0]
    )
    focus_mask = np.ones(len(df), dtype=bool)
    for c in group_cols:
        focus_mask &= (df[c].astype(str) == str(top_group[c]))
    focus_candidates = df[focus_mask].copy()

best_flow_mode = "nonlinear" if "nonlinear" in focus_candidates["flow_mode"].astype(str).unique() else focus_candidates["flow_mode"].iloc[0]
focus_df = focus_candidates[focus_candidates["flow_mode"].astype(str) == str(best_flow_mode)].copy()

focus_meta = {
    "system": str(focus_df["system"].iloc[0]),
    "task": str(focus_df["task"].iloc[0]),
    "forcing_mode": str(focus_df["forcing_mode"].iloc[0]),
    "flow_mode": str(focus_df["flow_mode"].iloc[0]),
}
print("FOCUS:", focus_meta)
display(focus_df.head())

In [ ]:
# Coefficient transport across k for the focus slice

rows_k = []
betas_k = {}

for kval in sorted(focus_df["k"].astype(float).unique()):
    sub = focus_df[focus_df["k"].astype(float) == float(kval)].copy()
    if len(sub) < 30:
        continue
    beta, pred, fit_stats = fit_template(sub)
    add_r2 = additive_fraction(sub)
    fb_gap = fb_gap_metric(sub, beta)

    row = {
        "k": kval,
        "system": focus_meta["system"],
        "task": focus_meta["task"],
        "forcing_mode": focus_meta["forcing_mode"],
        "flow_mode": focus_meta["flow_mode"],
        "additive_r2": add_r2,
        "fb_gap": fb_gap,
    }
    row.update(fit_stats)
    rows_k.append(row)
    betas_k[kval] = beta

coef_k_df = pd.DataFrame(rows_k).sort_values("k").reset_index(drop=True)
display(coef_k_df)

In [ ]:
# Plot coefficients vs k

plt.figure(figsize=(8, 5))
for term in TERM_NAMES[1:]:
    plt.plot(coef_k_df["k"], coef_k_df[f"coef_{term}"], marker="o", label=term)

plt.xlabel("k")
plt.ylabel("coefficient value")
plt.title("Coefficient transport across k")
plt.legend()
plt.tight_layout()
plt.show()

## Coefficient transport across forcing mode

In [ ]:
# Fix system/task/k/flow and compare forcing modes

k_anchor = 5 if 5 in focus_df["k"].astype(float).unique() else float(sorted(focus_df["k"].astype(float).unique())[0])

rows_force = []
for fmode in sorted(df["forcing_mode"].astype(str).unique()):
    sub = df[
        (df["system"].astype(str) == focus_meta["system"])
        & (df["task"].astype(str) == focus_meta["task"])
        & (df["flow_mode"].astype(str) == focus_meta["flow_mode"])
        & (df["forcing_mode"].astype(str) == fmode)
        & (df["k"].astype(float) == float(k_anchor))
    ].copy()
    if len(sub) < 30:
        continue
    beta, pred, fit_stats = fit_template(sub)
    row = {
        "forcing_mode": fmode,
        "k": k_anchor,
        "system": focus_meta["system"],
        "task": focus_meta["task"],
        "flow_mode": focus_meta["flow_mode"],
        "additive_r2": additive_fraction(sub),
        "fb_gap": fb_gap_metric(sub, beta),
    }
    row.update(fit_stats)
    rows_force.append(row)

coef_force_df = pd.DataFrame(rows_force).sort_values("forcing_mode").reset_index(drop=True)
display(coef_force_df)

In [ ]:
# Forcing-mode coefficient heatmap

force_terms = [f"coef_{t}" for t in TERM_NAMES]
heat = coef_force_df.set_index("forcing_mode")[force_terms]

plt.figure(figsize=(8, 4))
plt.imshow(heat.values, aspect="auto", cmap="coolwarm")
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(len(heat.columns)), [c.replace("coef_", "") for c in heat.columns], rotation=45, ha="right")
plt.colorbar(label="coefficient")
plt.title("Coefficient transport across forcing modes")
plt.tight_layout()
plt.show()

## Coefficient transport across task

In [ ]:
rows_task = []
for task in sorted(df["task"].astype(str).unique()):
    sub = df[
        (df["system"].astype(str) == focus_meta["system"])
        & (df["forcing_mode"].astype(str) == focus_meta["forcing_mode"])
        & (df["flow_mode"].astype(str) == focus_meta["flow_mode"])
        & (df["task"].astype(str) == task)
        & (df["k"].astype(float) == float(k_anchor))
    ].copy()
    if len(sub) < 30:
        continue
    beta, pred, fit_stats = fit_template(sub)
    row = {
        "task": task,
        "k": k_anchor,
        "system": focus_meta["system"],
        "forcing_mode": focus_meta["forcing_mode"],
        "flow_mode": focus_meta["flow_mode"],
        "additive_r2": additive_fraction(sub),
        "fb_gap": fb_gap_metric(sub, beta),
    }
    row.update(fit_stats)
    rows_task.append(row)

coef_task_df = pd.DataFrame(rows_task).sort_values("task").reset_index(drop=True)
display(coef_task_df)

In [ ]:
# Task coefficient heatmap

task_terms = [f"coef_{t}" for t in TERM_NAMES]
heat = coef_task_df.set_index("task")[task_terms]

plt.figure(figsize=(8, 4))
plt.imshow(heat.values, aspect="auto", cmap="coolwarm")
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(len(heat.columns)), [c.replace("coef_", "") for c in heat.columns], rotation=45, ha="right")
plt.colorbar(label="coefficient")
plt.title("Coefficient transport across task")
plt.tight_layout()
plt.show()

## Coefficient transport across system

In [ ]:
rows_system = []
for system in sorted(df["system"].astype(str).unique()):
    sub = df[
        (df["system"].astype(str) == system)
        & (df["task"].astype(str) == focus_meta["task"])
        & (df["forcing_mode"].astype(str) == focus_meta["forcing_mode"])
        & (df["flow_mode"].astype(str) == focus_meta["flow_mode"])
        & (df["k"].astype(float) == float(k_anchor))
    ].copy()
    if len(sub) < 30:
        continue
    beta, pred, fit_stats = fit_template(sub)
    row = {
        "system": system,
        "k": k_anchor,
        "task": focus_meta["task"],
        "forcing_mode": focus_meta["forcing_mode"],
        "flow_mode": focus_meta["flow_mode"],
        "additive_r2": additive_fraction(sub),
        "fb_gap": fb_gap_metric(sub, beta),
    }
    row.update(fit_stats)
    rows_system.append(row)

coef_system_df = pd.DataFrame(rows_system).sort_values("system").reset_index(drop=True)
display(coef_system_df)

In [ ]:
# System coefficient heatmap

sys_terms = [f"coef_{t}" for t in TERM_NAMES]
heat = coef_system_df.set_index("system")[sys_terms]

plt.figure(figsize=(8, 4))
plt.imshow(heat.values, aspect="auto", cmap="coolwarm")
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(len(heat.columns)), [c.replace("coef_", "") for c in heat.columns], rotation=45, ha="right")
plt.colorbar(label="coefficient")
plt.title("Coefficient transport across system")
plt.tight_layout()
plt.show()

## Universality transfer matrix

Fit coefficients on one regime and test them on another regime.
This asks whether the same law transfers, not just whether it fits locally.

In [ ]:
# Build regime table for the focus family varying only k and forcing mode first

regime_rows = []
for fmode in sorted(df["forcing_mode"].astype(str).unique()):
    for kval in sorted(df["k"].astype(float).unique()):
        sub = df[
            (df["system"].astype(str) == focus_meta["system"])
            & (df["task"].astype(str) == focus_meta["task"])
            & (df["flow_mode"].astype(str) == focus_meta["flow_mode"])
            & (df["forcing_mode"].astype(str) == fmode)
            & (df["k"].astype(float) == float(kval))
        ].copy()
        if len(sub) < 30:
            continue
        regime_id = f"{fmode}|k={int(kval) if float(kval).is_integer() else kval}"
        beta, pred, fit_stats = fit_template(sub)
        regime_rows.append({
            "regime_id": regime_id,
            "forcing_mode": fmode,
            "k": kval,
            "beta": beta,
            "sub": sub,
        })

regimes = pd.DataFrame(regime_rows)
display(regimes[["regime_id", "forcing_mode", "k"]])

In [ ]:
# Transfer R² matrix

regime_ids = regimes["regime_id"].tolist()
transfer = pd.DataFrame(index=regime_ids, columns=regime_ids, dtype=float)

for _, row_fit in regimes.iterrows():
    beta = row_fit["beta"]
    for _, row_test in regimes.iterrows():
        sub_test = row_test["sub"]
        pred_test = predict_with_beta(sub_test, beta)
        y_test = sub_test["predicted_flow"].to_numpy()
        transfer.loc[row_fit["regime_id"], row_test["regime_id"]] = r2_score(y_test, pred_test)

display(transfer.round(3))

In [ ]:
# Plot universality transfer matrix

plt.figure(figsize=(8, 6))
plt.imshow(
    transfer.values,
    aspect="auto",
    cmap="viridis",
    vmin=min(-0.2, float(np.nanmin(transfer.values))),
    vmax=float(np.nanmax(transfer.values)),
)
plt.xticks(range(len(transfer.columns)), transfer.columns, rotation=45, ha="right")
plt.yticks(range(len(transfer.index)), transfer.index)
plt.colorbar(label="transfer R²")
plt.title("Universality transfer matrix")
plt.tight_layout()
plt.show()

## Shared-law vs regime-specific law

Compare:

- one shared coefficient vector across all focus-family regimes
- one coefficient vector per regime

In [ ]:
# Shared vs regime-specific fit

all_focus_regimes = pd.concat(regimes["sub"].tolist(), ignore_index=True)
beta_shared, pred_shared, shared_stats = fit_template(all_focus_regimes)

shared_rows = []
for _, row in regimes.iterrows():
    sub = row["sub"]
    y = sub["predicted_flow"].to_numpy()
    pred_shared_sub = predict_with_beta(sub, beta_shared)
    pred_specific_sub = predict_with_beta(sub, row["beta"])
    shared_rows.append({
        "regime_id": row["regime_id"],
        "shared_r2": r2_score(y, pred_shared_sub),
        "specific_r2": r2_score(y, pred_specific_sub),
        "shared_rmse": math.sqrt(mean_squared_error(y, pred_shared_sub)),
        "specific_rmse": math.sqrt(mean_squared_error(y, pred_specific_sub)),
    })

shared_compare_df = pd.DataFrame(shared_rows)
display(shared_compare_df)

In [ ]:
# Shared vs specific error plot

plt.figure(figsize=(8, 5))
x = np.arange(len(shared_compare_df))
w = 0.36
plt.bar(x - w/2, shared_compare_df["shared_rmse"], width=w, label="shared law")
plt.bar(x + w/2, shared_compare_df["specific_rmse"], width=w, label="regime-specific")
plt.xticks(x, shared_compare_df["regime_id"], rotation=45, ha="right")
plt.ylabel("RMSE")
plt.title("Shared-law vs regime-specific fit")
plt.legend()
plt.tight_layout()
plt.show()

## Coefficient manifold

We stack all coefficient vectors and look for a low-dimensional geometry.

In [ ]:
# Coefficient matrix across broader regime set

coef_rows = []
group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, fit_stats = fit_template(sub.copy())
    row = {
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
        "additive_r2": additive_fraction(sub),
        "fb_gap": fb_gap_metric(sub, beta),
        "r2": fit_stats["r2"],
        "rmse": fit_stats["rmse"],
    }
    for name, coef in zip(TERM_NAMES, beta):
        row[f"coef_{name}"] = coef
    coef_rows.append(row)

coef_df = pd.DataFrame(coef_rows)
coef_cols = [f"coef_{t}" for t in TERM_NAMES]
display(coef_df.head())

In [ ]:
# PCA on coefficients

Xcoef = coef_df[coef_cols].to_numpy()
Xcoef_std = StandardScaler().fit_transform(Xcoef)
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(Xcoef_std)

coef_df["pc1"] = Z[:, 0]
coef_df["pc2"] = Z[:, 1]

print("Explained variance ratio:", pca.explained_variance_ratio_)
display(coef_df[["system", "task", "forcing_mode", "k", "flow_mode", "pc1", "pc2"]].head())

In [ ]:
# Plot coefficient manifold colored by forcing mode

plt.figure(figsize=(7, 5))
forcing_modes = sorted(coef_df["forcing_mode"].astype(str).unique())
for fmode in forcing_modes:
    sub = coef_df[coef_df["forcing_mode"].astype(str) == fmode]
    plt.scatter(sub["pc1"], sub["pc2"], label=fmode)

for _, row in coef_df.iterrows():
    kval = int(row["k"]) if float(row["k"]).is_integer() else row["k"]
    plt.text(row["pc1"], row["pc2"], f"k={kval}", fontsize=7, alpha=0.7)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Coefficient manifold across regimes")
plt.legend()
plt.tight_layout()
plt.show()

## Coefficients vs dynamical diagnostics

In [ ]:
# Scatter diagnostics

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(coef_df["coef_r"], coef_df["fb_gap"])
axes[0].set_xlabel("coef_r")
axes[0].set_ylabel("forward/backward gap")
axes[0].set_title("Restoring term vs asymmetry")

axes[1].scatter(coef_df["coef_c"], coef_df["additive_r2"])
axes[1].set_xlabel("coef_c")
axes[1].set_ylabel("additive R²")
axes[1].set_title("Drive term vs additive fraction")

axes[2].scatter(coef_df["coef_r c^2"], coef_df["fb_gap"])
axes[2].set_xlabel("coef_r c^2")
axes[2].set_ylabel("forward/backward gap")
axes[2].set_title("Interaction term vs asymmetry")

plt.tight_layout()
plt.show()

## Final summary table

Each row is a regime with:
- coefficient vector
- fit quality
- additive fraction
- forward/backward asymmetry

In [ ]:
# Verdicts

def verdict(row):
    if row["r2"] > 0.98 and row["additive_r2"] > 0.75 and row["fb_gap"] > 0.5:
        return "same family, strong transport"
    if row["r2"] > 0.98:
        return "same family, mild transport"
    if row["r2"] > 0.90:
        return "shared-law adequate"
    return "regime-specific correction needed"

coef_df["verdict"] = coef_df.apply(verdict, axis=1)

summary_cols = ["system", "task", "forcing_mode", "k", "flow_mode", "r2", "rmse", "additive_r2", "fb_gap"] + coef_cols + ["verdict"]
display(coef_df[summary_cols].head(20))

## Working conclusion

Notebook 47 tests whether the interpretable law from Notebook 46 is universal in structure and how its coefficients
transport across regimes.

A strong outcome is:

- the fixed sparse family remains adequate across regimes
- transfer performance stays good but not perfect
- regime-specific coefficients improve fit modestly
- coefficient vectors occupy a low-dimensional manifold

If that holds on your real exports, the next notebook is:

**Notebook 48 — regime reduction and meta-law for coefficient dynamics**

where the coefficient vector itself is modeled as a function of regime variables.